# 04 — Bottleneck graphs: serious over-squashing the path algebra resolves

Trees (NeighborsMatch) have **unique** paths, so kQ/I has nothing to collapse — that is why the quotient ties
the baselines there. To exhibit *serious* over-squashing that the path algebra can actually fix, we use a
**bottleneck chain**: `K` sources → `d` layers of `M` *interchangeable* nodes → one target.

Because each layer's nodes are interchangeable, the number of length-`(d+1)` walks source→target grows as
`~K·M^d` (a vanilla GNN must squash all of them into the target's fixed-width vector), while under kQ/I all
those walks are **functionally equivalent** and collapse to `~K`. The deeper `d`, the worse the squashing —
and the bigger the kQ/I advantage.

**Task (bottlenecked retrieval):** the target must output the value of the source whose key matches a query
placed on the target. The needed signal sits `d+1` hops away behind the bottleneck.

All runs are tracked in **MLflow** (SQLite backend). View them with:
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

## 1. The over-squashing knob: raw vs. effective multiplicity

Confirm that raw walk multiplicity into the target explodes with depth while the kQ/I-effective count stays
flat. This is the structural reason the quotient should help.

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators

g = torch.Generator().manual_seed(0); K, M = 5, 4
rows = []
for depth in [1, 2, 3, 4]:
    data = make_bottleneck_graph(K, M, depth, g)
    ei, N = data.edge_index.numpy(), data.x.size(0)
    t = int(data.root_mask.nonzero()[0]); L = depth + 1
    raw, eff = walk_operators(ei, N, max_length=L)
    rows.append({'depth': depth, 'nodes': N,
                 'raw_mult': float(raw[L-1][:, t].sum()),
                 'eff_mult': float(eff[L-1][:, t].sum())})
knob = pd.DataFrame(rows); print(knob.to_string(index=False))

## 2. Run the sweep (tracked in MLflow)

Five models — GCN/GAT/GIN baselines, `walkraw` (multi-hop over raw `A^g`, the architecture-matched ablation),
and `quotient` (multi-hop over effective kQ/I `M_g`). The `quotient` − `walkraw` gap is the pure kQ/I effect.

In [ ]:
import yaml, functools
from oversquash.mlflow_runner import run_sweep_mlflow
from oversquash.data_bottleneck import bottleneck_dataset

cfg = yaml.safe_load(open('../configs/bottleneck.yaml'))
cfg['results_dir'] = '../results'
ds = functools.partial(bottleneck_dataset, K=5, M=4)
df = run_sweep_mlflow(cfg, ds, experiment='bottleneck-oversquash',
                      tracking_uri='sqlite:///../mlflow.db', run_name='nb04_K5_M4')
df.pivot(index='depth', columns='model', values='val_acc').round(3)

## 3. Headline figure: accuracy vs. bottleneck depth

Expect vanilla GCN/GIN at chance (1/K = 0.2), and `quotient` well above — with `quotient` ≥ `walkraw`
isolating the algebra's contribution.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for name, gg in df.groupby('model'):
    gg = gg.sort_values('depth')
    ax.plot(gg['depth'], gg['val_acc'], marker='o', label=name)
ax.axhline(1/5, color='grey', ls='--', lw=1, label='chance (1/K)')
ax.set_xlabel('bottleneck depth $d$  (over-squashing radius)')
ax.set_ylabel('accuracy'); ax.set_xticks([1,2,3,4])
ax.set_title('Bottleneck retrieval: kQ/I vs. baselines'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

## 4. Robustness: multiple seeds

A single seed can show spurious inversions (e.g. `walkraw` > `quotient` at one depth). Average over seeds
before drawing conclusions. The multi-seed table is written to `results/tables/bottleneck_multiseed.csv`.

In [ ]:
from pathlib import Path
p = Path('../results/tables/bottleneck_multiseed.csv')
if p.exists():
    big = pd.read_csv(p)
    mean = big.groupby(['depth','model'])['val_acc'].mean().reset_index()
    std = big.groupby(['depth','model'])['val_acc'].std().reset_index()
    print('MEAN over seeds:'); print(mean.pivot(index='depth', columns='model', values='val_acc').round(3))
    print('\nSTD over seeds:'); print(std.pivot(index='depth', columns='model', values='val_acc').round(3))
else:
    print('Run the multi-seed sweep first (see repo: results/tables/bottleneck_multiseed.csv).')